<a href="https://colab.research.google.com/github/m-edal/Earth-Env-DS-MSc-Course/blob/main/labs/W9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Climate Time Series Forecasting with PyTorch

**Lab Duration:** ~2-3 hours

## Overview

In this lab, you will learn how to use **PyTorch** — one of the most popular deep learning frameworks — to build neural networks that forecast future climate variables from historical time series data.

We will work with output from the **Community Earth System Model Large Ensemble (CESM-LENS)** project. This dataset contains daily climate simulations from 2006 to 2080 under the RCP8.5 emission scenario. Multiple ensemble members (runs with slightly different initial conditions) capture the range of natural climate variability.

**Our goal:** Given a window of past daily climate observations, predict the next day's **urban maximum temperature** (`TREFMXAV_U`).

### What You Will Learn

| Section | Topic |
|---------|-------|
| Part 0 | Setup & Data Download |
| Part 1 | PyTorch Fundamentals — Tensors, Datasets, and Training Loops |
| Part 2 | Data Exploration & Preprocessing |
| Part 3 | Sliding Window — Turning Time Series into Supervised Learning |
| Part 4 | Baseline Model — Simple "Repeat Last Value" |
| Part 5 | Fully Connected (Dense) Neural Network |
| Part 6 | LSTM (Long Short-Term Memory) Network |
| Part 7 | Model Comparison & Analysis |

### Prerequisites

- Basic Python and NumPy
- Familiarity with traditional ML concepts (train/test split, loss functions, overfitting)
- **No prior PyTorch or deep learning experience required**

### Notation

Throughout this notebook:
- 🟢 **Pre-filled code**: Run as-is to see results.
- 🔴 **TODO / Solution Area**: You need to write code here. A solution is hidden below each task.
- 💡 **Tip**: Helpful hints and explanations.

---
## Part 0: Setup & Data Download

First, let's install the required packages and download the data.

In [ ]:
# Install required packages (run this cell once)
!pip install -q torch numpy pandas matplotlib scikit-learn xarray netCDF4

In [ ]:
# Download the CESM-LENS ensemble data from Dropbox
!curl -L -o cesm_data.zip "https://www.dropbox.com/scl/fo/dmabz9pf3167l62612h5b/h?rlkey=ge8u486w7w7vq8vnpr2f1fvag&dl=1"
!unzip -o cesm_data.zip -d cesm_data/
!ls cesm_data/

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---
## Part 1: PyTorch Fundamentals

Before building models, let's learn the essential building blocks of PyTorch.

### 1.1 Tensors — The Core Data Structure

A **tensor** is PyTorch's version of a NumPy array, but with two superpowers:
1. It can live on a **GPU** for fast parallel computation.
2. It supports **automatic differentiation** (autograd) — PyTorch can automatically compute gradients, which is how neural networks learn.

If you're familiar with NumPy, tensors will feel very natural.

In [ ]:
# Creating tensors — just like NumPy!
a = torch.tensor([1.0, 2.0, 3.0])
print(f"Tensor a: {a}")
print(f"Shape:    {a.shape}")
print(f"Dtype:    {a.dtype}")

# From NumPy
np_array = np.array([[1, 2], [3, 4]], dtype=np.float32)
b = torch.from_numpy(np_array)
print(f"\nTensor b (from NumPy):\n{b}")

# Common creation functions
zeros = torch.zeros(2, 3)        # 2x3 matrix of zeros
ones = torch.ones(2, 3)          # 2x3 matrix of ones
rand = torch.randn(2, 3)         # 2x3 matrix of random normal values
print(f"\nRandom tensor:\n{rand}")

In [ ]:
# Tensor operations — same syntax as NumPy
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])

print(f"x + y = {x + y}")           # Element-wise addition
print(f"x * y = {x * y}")           # Element-wise multiplication
print(f"x.mean() = {x.mean()}")     # Mean
print(f"x.sum()  = {x.sum()}")      # Sum

# Reshaping
m = torch.arange(12)               # [0, 1, 2, ..., 11]
m_reshaped = m.reshape(3, 4)       # Reshape to 3x4
print(f"\nReshaped:\n{m_reshaped}")

### 1.2 Automatic Differentiation (Autograd)

This is the key feature that makes deep learning possible. When you set `requires_grad=True`, PyTorch tracks every operation on a tensor. You can then call `.backward()` to automatically compute gradients.

**Why does this matter?** Neural networks learn by:
1. Making a prediction (forward pass)
2. Computing the error (loss)
3. Computing how to adjust each weight to reduce the error (**gradients**, via backward pass)
4. Updating the weights (optimizer step)

In [ ]:
# Simple example: compute gradient of y = x^2 at x = 3
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2             # y = 9
y.backward()           # Compute dy/dx
print(f"x = {x.item()}, y = x² = {y.item()}")
print(f"dy/dx = 2x = {x.grad.item()}")  # Should be 2*3 = 6

### 1.3 `nn.Module` — Building Neural Network Layers

In PyTorch, every neural network is a Python class that inherits from `nn.Module`. You define:
- `__init__`: Create the layers
- `forward`: Define how data flows through the layers

In [ ]:
# A simple single-layer network: input -> linear transformation -> output
class SimpleNet(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)  # y = Wx + b

    def forward(self, x):
        return self.linear(x)

# Create and test
net = SimpleNet(input_size=3, output_size=1)
sample_input = torch.randn(1, 3)  # 1 sample, 3 features
output = net(sample_input)
print(f"Input shape:  {sample_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Output value: {output.item():.4f}")

# Count parameters
total_params = sum(p.numel() for p in net.parameters())
print(f"Total parameters: {total_params}")
# nn.Linear(3, 1) has 3 weights + 1 bias = 4 parameters

### 1.4 The Training Loop

Unlike scikit-learn's `.fit()`, PyTorch gives you full control over the training loop. This is more work but offers much more flexibility. Here's the pattern you'll use throughout this lab:

```python
for epoch in range(num_epochs):       # Repeat over the full dataset
    for X_batch, y_batch in dataloader: # Iterate over mini-batches
        predictions = model(X_batch)   # 1. Forward pass
        loss = loss_fn(predictions, y_batch)  # 2. Compute loss
        optimizer.zero_grad()          # 3. Clear old gradients
        loss.backward()                # 4. Compute new gradients
        optimizer.step()               # 5. Update weights
```

💡 **Tip:** Think of this like gradient descent in traditional ML, but applied to a neural network with potentially millions of parameters.

---
## Part 2: Data Exploration & Preprocessing

### 2.1 Understanding the Data

Our data comes from **CESM-LENS (Community Earth System Model Large Ensemble)**. It is a climate model simulation, not observational data. The files are in **NetCDF** format (`.nc`), a standard format in climate science.

Each file represents one **ensemble member** — the same simulation run with slightly perturbed initial conditions. This captures the **internal variability** of the climate system.

**Variables (all daily values):**

| Variable | Description | Unit |
|----------|-------------|------|
| `TREFMXAV_U` | Urban daily maximum of average 2m temperature | K |
| `TREFHT` | Reference height (2m) temperature | K |
| `FLNS` | Net longwave flux at surface | W/m² |
| `FSNS` | Net shortwave flux at surface | W/m² |
| `PRECT` | Total precipitation rate | m/s |
| `PRSN` | Snowfall rate | m/s |
| `QBOT` | Near-surface specific humidity | kg/kg |
| `UBOT` | Lowest model level zonal (east-west) wind | m/s |
| `VBOT` | Lowest model level meridional (north-south) wind | m/s |

In [ ]:
# Load one ensemble member to explore
import glob
nc_files = sorted(glob.glob('cesm_data/*.nc'))
print(f"Found {len(nc_files)} ensemble member files:")
for f in nc_files:
    print(f"  {f}")

# Open the first file
ds = xr.open_dataset(nc_files[0])
print(f"\nDataset dimensions: {dict(ds.dims)}")
print(f"Time range: {str(ds.time.values[0])[:10]} to {str(ds.time.values[-1])[:10]}")
print(f"Latitude range: {ds.lat.values.min():.2f} to {ds.lat.values.max():.2f}")
print(f"Longitude range: {ds.lon.values.min():.2f} to {ds.lon.values.max():.2f}")
print(f"\nVariables: {list(ds.data_vars)}")
ds

### 2.2 Extract a Single Grid Point as a Time Series

The data has 3 spatial dimensions (time, lat, lon). For our time series lab, we will pick **one grid point** (one latitude-longitude location) and extract the daily time series.

💡 **Why one grid point?** This simplifies the problem to a pure time series forecasting task, which is the focus of this lab. Spatial modeling (e.g., ConvLSTM) would be a more advanced topic.

In [ ]:
# Pick a grid point (roughly central in the domain)
# We use .sel with method='nearest' to snap to the closest grid cell
target_lat = ds.lat.values[len(ds.lat)//2]
target_lon = ds.lon.values[len(ds.lon)//2]
print(f"Selected grid point: lat={target_lat:.2f}, lon={target_lon:.2f}")

# Extract time series for all variables at this grid point
features_list = ['FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']
target_var = 'TREFMXAV_U'
all_vars = features_list + [target_var]

# Load from the first ensemble member
ds_point = ds[all_vars].sel(lat=target_lat, lon=target_lon, method='nearest')
df = ds_point.to_dataframe().reset_index()

# Convert time to datetime if needed
df['time'] = pd.to_datetime(df['time'])
df = df.set_index('time').drop(columns=['lat', 'lon'], errors='ignore')

print(f"\nDataFrame shape: {df.shape}")
print(f"Time range: {df.index[0]} to {df.index[-1]}")
df.head()

In [ ]:
# Quick statistics
df.describe().round(4)

### 2.3 Visualize the Data

In [ ]:
# Plot the target variable: urban maximum temperature
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Full time series
axes[0].plot(df.index, df[target_var] - 273.15, linewidth=0.3, alpha=0.7)
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title(f'Urban Daily Max Temperature ({target_var}) — Full Time Series')
axes[0].grid(True, alpha=0.3)

# Net shortwave radiation (solar energy hitting surface)
axes[1].plot(df.index, df['FSNS'], linewidth=0.3, alpha=0.7, color='orange')
axes[1].set_ylabel('FSNS (W/m²)')
axes[1].set_title('Net Shortwave Flux at Surface')
axes[1].grid(True, alpha=0.3)

# Precipitation
axes[2].plot(df.index, df['PRECT'] * 86400 * 1000, linewidth=0.3, alpha=0.7, color='blue')
axes[2].set_ylabel('Precip (mm/day)')
axes[2].set_title('Total Precipitation Rate')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Zoom into one year to see seasonal patterns
year_slice = df.loc['2020':'2020']
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(year_slice.index, year_slice[target_var] - 273.15, marker='.', markersize=2, linewidth=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Urban Max Temperature — Year 2020 (One Year Zoom)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

💡 **Observation:** You should see a clear **seasonal cycle** (warm summers, cold winters) and significant **day-to-day variability**. This is exactly the kind of pattern that deep learning models can learn from.

### 2.4 Train / Validation / Test Split

For time series, we **cannot** randomly shuffle the data (that would leak future information). Instead, we split **chronologically**:

```
|--- Training (70%) ---|--- Validation (15%) ---|--- Test (15%) ---|
2006                   ~2058                    ~2069             2080
```

In [ ]:
# Chronological split
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

df_train = df.iloc[:train_end]
df_val = df.iloc[train_end:val_end]
df_test = df.iloc[val_end:]

print(f"Training:   {len(df_train)} days ({df_train.index[0].date()} to {df_train.index[-1].date()})")
print(f"Validation: {len(df_val)} days ({df_val.index[0].date()} to {df_val.index[-1].date()})")
print(f"Test:       {len(df_test)} days ({df_test.index[0].date()} to {df_test.index[-1].date()})")

### 2.5 Feature Normalization

Neural networks train much better when features are on similar scales. We use **StandardScaler** (zero mean, unit variance) fitted **only on training data** to prevent data leakage.

💡 **Important:** We fit the scaler on training data only, then transform validation and test data with the same scaler. This mimics a real deployment scenario where future data statistics are unknown.

In [ ]:
# Fit scaler on training data only
scaler = StandardScaler()
scaler.fit(df_train[all_vars])

# Transform all splits
train_scaled = scaler.transform(df_train[all_vars])
val_scaled = scaler.transform(df_val[all_vars])
test_scaled = scaler.transform(df_test[all_vars])

# Convert to numpy arrays
# Columns: [features..., target] — target is the last column
print(f"Scaled training data shape: {train_scaled.shape}")
print(f"Feature means (train, should be ~0): {train_scaled.mean(axis=0).round(4)}")
print(f"Feature stds  (train, should be ~1): {train_scaled.std(axis=0).round(4)}")

---
## Part 3: Sliding Window — Turning Time Series into Supervised Learning

### The Key Idea

A raw time series is just a sequence of values: $[x_1, x_2, x_3, \ldots, x_T]$

To train a neural network, we need **(input, output) pairs**. The **sliding window** approach creates these by slicing the time series:

```
Window size = 7 (use 7 days of history to predict the next day)

Input Window              → Target
[day1, day2, ..., day7]   → day8
[day2, day3, ..., day8]   → day9
[day3, day4, ..., day9]   → day10
...                       → ...
```

Each input is a matrix of shape `(window_size, num_features)` and each target is a single value (the temperature we want to predict).

💡 **Analogy:** Imagine looking at the past week's weather report and predicting tomorrow's temperature. The "past week" is your input window.

In [ ]:
# Visualize the sliding window concept
fig, ax = plt.subplots(figsize=(12, 3))

# Show a small portion of the temperature data
demo_data = train_scaled[:30, -1]  # last column = target
ax.plot(range(30), demo_data, 'k.-', label='Temperature')

# Highlight one window
window_size = 7
start = 5
ax.axvspan(start, start + window_size - 1, alpha=0.2, color='blue', label='Input window (7 days)')
ax.axvspan(start + window_size, start + window_size, alpha=0.4, color='red', label='Target (day 8)')
ax.scatter([start + window_size], [demo_data[start + window_size]], color='red', s=100, zorder=5)

ax.set_xlabel('Day Index')
ax.set_ylabel('Scaled Temperature')
ax.set_title('Sliding Window: Use 7 days to predict the next day')
ax.legend()
plt.tight_layout()
plt.show()

### 3.1 Create a PyTorch Dataset

PyTorch's `Dataset` class is a standard way to organize your data. It needs two methods:
- `__len__()`: How many samples?
- `__getitem__(idx)`: Return the idx-th (input, target) pair

In [ ]:
class TimeSeriesDataset(Dataset):
    """
    Creates (input_window, target) pairs from a scaled time series array.

    Parameters:
        data: numpy array of shape (num_timesteps, num_features)
              The last column is assumed to be the target variable.
        window_size: int, number of past time steps to use as input
    """
    def __init__(self, data, window_size):
        self.data = torch.FloatTensor(data)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        # Input: all features for the window
        x = self.data[idx : idx + self.window_size]          # shape: (window_size, num_features)
        # Target: the target variable (last column) at the next time step
        y = self.data[idx + self.window_size, -1]            # shape: scalar
        return x, y

In [ ]:
# Create datasets with a 7-day window
WINDOW_SIZE = 7
BATCH_SIZE = 64

train_dataset = TimeSeriesDataset(train_scaled, WINDOW_SIZE)
val_dataset = TimeSeriesDataset(val_scaled, WINDOW_SIZE)
test_dataset = TimeSeriesDataset(test_scaled, WINDOW_SIZE)

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")

# Inspect one sample
x_sample, y_sample = train_dataset[0]
print(f"\nInput shape:  {x_sample.shape}  → (window_size={WINDOW_SIZE}, num_features={x_sample.shape[1]})")
print(f"Target shape: {y_sample.shape}  → scalar (next day's temperature)")

In [ ]:
# Create DataLoaders for efficient batching
# shuffle=True for training (helps with convergence), False for validation/test
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Check one batch
x_batch, y_batch = next(iter(train_loader))
print(f"Batch input shape:  {x_batch.shape}  → (batch_size, window_size, num_features)")
print(f"Batch target shape: {y_batch.shape}  → (batch_size,)")

---
## Part 4: Baseline Model — "Repeat Last Value"

Before building fancy neural networks, we need a simple baseline to compare against. The simplest time series forecast is: **tomorrow's temperature = today's temperature**.

This is surprisingly hard to beat for daily data! Any deep learning model that can't outperform this baseline is not learning anything useful.

In [ ]:
def evaluate_baseline(dataset):
    """Baseline: predict that tomorrow = today (last value in window)."""
    preds = []
    targets = []
    for i in range(len(dataset)):
        x, y = dataset[i]
        # Prediction = last time step's target variable (last column)
        pred = x[-1, -1].item()
        preds.append(pred)
        targets.append(y.item())
    preds = np.array(preds)
    targets = np.array(targets)
    mae = np.mean(np.abs(preds - targets))
    mse = np.mean((preds - targets) ** 2)
    return mae, mse, preds, targets

baseline_mae, baseline_mse, baseline_preds, baseline_targets = evaluate_baseline(test_dataset)

# Convert MAE back to Kelvin (un-normalize)
target_idx = all_vars.index(target_var)
target_std = scaler.scale_[target_idx]
baseline_mae_K = baseline_mae * target_std

print(f"Baseline (Repeat Last Value):")
print(f"  MAE (normalized): {baseline_mae:.4f}")
print(f"  MSE (normalized): {baseline_mse:.4f}")
print(f"  MAE (Kelvin):     {baseline_mae_K:.2f} K ≈ {baseline_mae_K:.2f} °C")

---
## Part 5: Fully Connected (Dense) Neural Network

Our first deep learning model is a simple **feedforward network** (also called a dense or fully connected network). It takes the flattened input window and passes it through several layers of linear transformations with non-linear activations.

```
Input: (window_size × num_features) → Flatten → Linear → ReLU → Linear → ReLU → Linear → Output (1)
```

### 5.1 Define the Model

🔴 **TODO:** Complete the `DenseModel` class below. You need to:
1. Define the layers in `__init__`
2. Implement the `forward` method

In [ ]:
class DenseModel(nn.Module):
    """
    A simple feedforward (dense) network for time series prediction.
    Flattens the input window and passes through fully connected layers.
    """
    def __init__(self, window_size, num_features, hidden_size=64):
        super().__init__()
        input_dim = window_size * num_features  # Flatten window into a vector

        # ============================================================
        # TODO: Define 3 linear layers with ReLU activations
        #
        # Architecture:
        #   Linear(input_dim → hidden_size) → ReLU
        #   Linear(hidden_size → hidden_size) → ReLU
        #   Linear(hidden_size → 1)
        #
        # Hint: Use nn.Sequential to chain layers together.
        #       nn.ReLU() is the activation function.
        # ============================================================

        self.network = None  # Replace this line

        # ============================================================

    def forward(self, x):
        """
        x shape: (batch_size, window_size, num_features)
        output shape: (batch_size,)
        """
        # ============================================================
        # TODO: Flatten x from (batch, window, features) to (batch, window*features)
        #       Then pass through self.network
        #       Return output squeezed to shape (batch_size,)
        #
        # Hint: x.reshape(x.shape[0], -1) flattens the last two dimensions
        #       output.squeeze(-1) removes the last dimension of size 1
        # ============================================================

        pass  # Replace this line

        # ============================================================

In [ ]:
# Test your model (run this to verify it works)
num_features = train_scaled.shape[1]
dense_model = DenseModel(WINDOW_SIZE, num_features, hidden_size=64).to(device)

# Quick forward pass test
test_input = torch.randn(4, WINDOW_SIZE, num_features).to(device)
test_output = dense_model(test_input)
print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}  ← should be (4,)")
print(f"\nModel architecture:")
print(dense_model)
total_params = sum(p.numel() for p in dense_model.parameters())
print(f"\nTotal parameters: {total_params:,}")

### 5.2 Training Infrastructure

Let's write reusable training and evaluation functions. These will be used for all our models.

In [ ]:
def train_one_epoch(model, dataloader, loss_fn, optimizer, device):
    """Train the model for one epoch and return average loss."""
    model.train()  # Set model to training mode
    total_loss = 0
    num_batches = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        predictions = model(X_batch)            # Forward pass
        loss = loss_fn(predictions, y_batch)     # Compute loss

        optimizer.zero_grad()                    # Clear old gradients
        loss.backward()                          # Backpropagation
        optimizer.step()                         # Update weights

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches


def evaluate(model, dataloader, loss_fn, device):
    """Evaluate the model and return average loss and MAE."""
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    total_mae = 0
    num_batches = 0

    with torch.no_grad():  # Disable gradient computation for efficiency
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch)

            total_loss += loss.item()
            total_mae += torch.mean(torch.abs(predictions - y_batch)).item()
            num_batches += 1

    return total_loss / num_batches, total_mae / num_batches


def get_predictions(model, dataloader, device):
    """Get all predictions and targets from a dataloader."""
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            preds = model(X_batch)
            all_preds.append(preds.cpu().numpy())
            all_targets.append(y_batch.numpy())

    return np.concatenate(all_preds), np.concatenate(all_targets)

### 5.3 Train the Dense Model

🔴 **TODO:** Complete the training loop below. You need to:
1. Choose a loss function (hint: MSE is standard for regression)
2. Choose an optimizer (hint: Adam is a good default)
3. Call the training and evaluation functions for each epoch

In [ ]:
# Re-create the model (fresh weights)
dense_model = DenseModel(WINDOW_SIZE, num_features, hidden_size=64).to(device)

NUM_EPOCHS = 20
LEARNING_RATE = 1e-3

# ============================================================
# TODO: Define loss function and optimizer
#
# loss_fn = ???     # Hint: nn.MSELoss()
# optimizer = ???   # Hint: torch.optim.Adam(model.parameters(), lr=...)
# ============================================================

loss_fn = None    # Replace this line
optimizer = None  # Replace this line

# ============================================================

# Training loop
train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
    # ============================================================
    # TODO: Call train_one_epoch and evaluate functions
    #
    # train_loss = train_one_epoch(???)
    # val_loss, val_mae = evaluate(???)
    # ============================================================

    train_loss = 0  # Replace this line
    val_loss, val_mae = 0, 0  # Replace this line

    # ============================================================

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val MAE: {val_mae:.4f}")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Training Loss')
ax.plot(val_losses, label='Validation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Dense Model — Training Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
dense_test_loss, dense_test_mae = evaluate(dense_model, test_loader, loss_fn, device)
dense_test_mae_K = dense_test_mae * target_std
print(f"Dense Model — Test MAE: {dense_test_mae:.4f} (normalized), {dense_test_mae_K:.2f} K")
print(f"Baseline     — Test MAE: {baseline_mae:.4f} (normalized), {baseline_mae_K:.2f} K")
print(f"Improvement: {(1 - dense_test_mae / baseline_mae) * 100:.1f}%")

---
## Part 6: LSTM (Long Short-Term Memory) Network

### Why LSTM?

The Dense model treats the input window as a flat vector — it ignores the **temporal order** of the days. An LSTM is specifically designed to process sequences step-by-step, maintaining a **hidden state** that acts as a memory of what it has seen so far.

```
Day 1 → [LSTM Cell] → hidden_1
Day 2 → [LSTM Cell] → hidden_2  (uses hidden_1 as context)
Day 3 → [LSTM Cell] → hidden_3  (uses hidden_2 as context)
...
Day 7 → [LSTM Cell] → hidden_7  (uses hidden_6 as context)
                         ↓
                    Linear → prediction
```

The LSTM learns **which information from the past to remember and which to forget**, making it powerful for time series with complex dependencies.

### 6.1 Define the LSTM Model

🔴 **TODO:** Complete the LSTM model. The key difference from the Dense model is that we use `nn.LSTM` instead of flattening.

In [ ]:
class LSTMModel(nn.Module):
    """
    LSTM network for time series prediction.
    Processes the input sequence step-by-step, then uses
    the final hidden state to make a prediction.
    """
    def __init__(self, num_features, hidden_size=64, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # ============================================================
        # TODO: Define the LSTM layer and a linear output layer
        #
        # self.lstm = nn.LSTM(
        #     input_size=???,       # Number of features at each time step
        #     hidden_size=???,      # Size of the hidden state
        #     num_layers=???,       # Number of stacked LSTM layers
        #     batch_first=True      # Input shape: (batch, seq_len, features)
        # )
        # self.fc = nn.Linear(???, 1)   # Map final hidden state to prediction
        # ============================================================

        self.lstm = None  # Replace this line
        self.fc = None    # Replace this line

        # ============================================================

    def forward(self, x):
        """
        x shape: (batch_size, window_size, num_features)
        output shape: (batch_size,)
        """
        # ============================================================
        # TODO:
        # 1. Pass x through self.lstm → returns (output, (hidden, cell))
        #    output shape: (batch, seq_len, hidden_size)
        # 2. Take the last time step's output: output[:, -1, :]
        # 3. Pass through self.fc
        # 4. Squeeze to (batch_size,)
        # ============================================================

        pass  # Replace this line

        # ============================================================

In [ ]:
# Test the LSTM model
lstm_model = LSTMModel(num_features, hidden_size=64, num_layers=1).to(device)

test_input = torch.randn(4, WINDOW_SIZE, num_features).to(device)
test_output = lstm_model(test_input)
print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}  ← should be (4,)")
print(f"\nModel architecture:")
print(lstm_model)
total_params = sum(p.numel() for p in lstm_model.parameters())
print(f"\nTotal parameters: {total_params:,}")

### 6.2 Train the LSTM

🔴 **TODO:** Train the LSTM model using the same pattern as the Dense model. You can reuse the `train_one_epoch` and `evaluate` functions.

In [ ]:
# Re-create model with fresh weights
lstm_model = LSTMModel(num_features, hidden_size=64, num_layers=1).to(device)

NUM_EPOCHS = 20
LEARNING_RATE = 1e-3

# ============================================================
# TODO: Define loss function and optimizer for the LSTM
# ============================================================

loss_fn = None    # Replace this line
optimizer = None  # Replace this line

# ============================================================

# Training loop
lstm_train_losses = []
lstm_val_losses = []

for epoch in range(NUM_EPOCHS):
    # ============================================================
    # TODO: Train and evaluate the LSTM model
    # ============================================================

    train_loss = 0  # Replace this line
    val_loss, val_mae = 0, 0  # Replace this line

    # ============================================================

    lstm_train_losses.append(train_loss)
    lstm_val_losses.append(val_loss)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val MAE: {val_mae:.4f}")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lstm_train_losses, label='Training Loss')
ax.plot(lstm_val_losses, label='Validation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM Model — Training Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
lstm_test_loss, lstm_test_mae = evaluate(lstm_model, test_loader, loss_fn, device)
lstm_test_mae_K = lstm_test_mae * target_std
print(f"LSTM Model   — Test MAE: {lstm_test_mae:.4f} (normalized), {lstm_test_mae_K:.2f} K")
print(f"Dense Model  — Test MAE: {dense_test_mae:.4f} (normalized), {dense_test_mae_K:.2f} K")
print(f"Baseline     — Test MAE: {baseline_mae:.4f} (normalized), {baseline_mae_K:.2f} K")

---
## Part 7: Model Comparison & Analysis

### 7.1 Summary Table

In [ ]:
# Build comparison table
results = pd.DataFrame({
    'Model': ['Baseline (Repeat Last)', 'Dense NN', 'LSTM'],
    'Test MAE (normalized)': [baseline_mae, dense_test_mae, lstm_test_mae],
    'Test MAE (K)': [baseline_mae_K, dense_test_mae_K, lstm_test_mae_K],
    'Test MSE (normalized)': [baseline_mse, dense_test_loss, lstm_test_loss],
})
results['Improvement vs Baseline'] = (
    (1 - results['Test MAE (normalized)'] / baseline_mae) * 100
).round(1).astype(str) + '%'
results.iloc[0, -1] = '—'

print("="*70)
print("MODEL COMPARISON")
print("="*70)
print(results.to_string(index=False))

### 7.2 Visualize Predictions

In [ ]:
# Get predictions from all models
dense_preds, targets = get_predictions(dense_model, test_loader, device)
lstm_preds, _ = get_predictions(lstm_model, test_loader, device)

# Un-normalize to get real temperatures (Kelvin → Celsius)
target_mean = scaler.mean_[target_idx]

targets_C = targets * target_std + target_mean - 273.15
dense_preds_C = dense_preds * target_std + target_mean - 273.15
lstm_preds_C = lstm_preds * target_std + target_mean - 273.15
baseline_preds_C = baseline_preds * target_std + target_mean - 273.15

# Plot a 60-day window for clarity
plot_range = slice(100, 160)

fig, ax = plt.subplots(figsize=(14, 5))
days = np.arange(len(targets_C[plot_range]))

ax.plot(days, targets_C[plot_range], 'k-', linewidth=2, label='Actual', alpha=0.8)
ax.plot(days, baseline_preds_C[plot_range], 's--', markersize=3, linewidth=1, label='Baseline', alpha=0.7)
ax.plot(days, dense_preds_C[plot_range], '^--', markersize=3, linewidth=1, label='Dense NN', alpha=0.7)
ax.plot(days, lstm_preds_C[plot_range], 'o--', markersize=3, linewidth=1, label='LSTM', alpha=0.7)

ax.set_xlabel('Day (relative to test period window)')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Test Set Predictions — 60-Day Window')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: predicted vs actual
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, preds, name in zip(axes,
                           [baseline_preds_C, dense_preds_C, lstm_preds_C],
                           ['Baseline', 'Dense NN', 'LSTM']):
    ax.scatter(targets_C, preds, alpha=0.1, s=5)
    lims = [targets_C.min(), targets_C.max()]
    ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
    ax.set_xlabel('Actual (°C)')
    ax.set_ylabel('Predicted (°C)')
    ax.set_title(name)
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicted vs Actual Temperature', fontsize=13)
plt.tight_layout()
plt.show()

### 7.3 Bonus: Using Ensemble Members

We downloaded 6 ensemble members but only used one for training. In practice, ensemble members can be used to:
1. **Augment training data** — train on multiple members for a more robust model.
2. **Quantify uncertainty** — predict on each member and look at the spread.

🔴 **TODO (Optional Challenge):** Load all 6 ensemble files, extract the same grid point, concatenate them, and re-train the LSTM. Does more data help?

In [ ]:
# ============================================================
# OPTIONAL CHALLENGE: Train on multiple ensemble members
#
# Steps:
# 1. Loop over nc_files, extract the same grid point from each
# 2. Concatenate the DataFrames (or keep them separate for train/val/test)
# 3. Apply the same normalization and windowing
# 4. Train a new LSTM model and compare
#
# Hint: A simple approach is to use ensemble members 003-007 for
#       training and member 008 for testing.
# ============================================================

# Your code here


---
## Summary & Key Takeaways

In this lab, you have learned:

1. **PyTorch Fundamentals** — Tensors, autograd, `nn.Module`, and the training loop.

2. **Time Series as Supervised Learning** — The sliding window approach converts a raw sequence into (input, target) pairs suitable for standard ML/DL models.

3. **Baseline Matters** — The "repeat last value" baseline is simple but effective. Always compare your models against it.

4. **Dense vs LSTM** — A dense network treats the input as an unordered vector, while an LSTM processes it step-by-step and can learn temporal dependencies.

5. **Climate Data Nuances** — CESM-LENS ensemble data provides multiple realizations of the climate system, which can be leveraged for data augmentation and uncertainty quantification.

### Ideas for Further Exploration

- **Tune hyperparameters:** Try different `WINDOW_SIZE` (14, 30 days), `hidden_size`, `num_layers`, learning rates.
- **Add a 1D CNN model:** Use `nn.Conv1d` to detect local patterns in the time series (similar to the TensorFlow tutorial).
- **Multi-step forecasting:** Predict not just the next day, but the next 7 days (multi-output).
- **Train on all ensemble members:** Use the optional challenge above.
- **Spatial modeling:** Use multiple grid points or a ConvLSTM to exploit spatial correlations.

---
*This lab was adapted from the [TensorFlow Time Series Forecasting Tutorial](https://www.tensorflow.org/tutorials/structured_data/time_series) and uses CESM-LENS climate simulation data.*